# Stellan-1 Max · HumanEval on a free Kaggle machine

Runs the same test as the repository's **Max benchmark** workflow: all 164 HumanEval problems, pass@1, greedy, each answer run against the official tests, with the app's own llama.cpp settings.

**Before running** (right-hand panel, **Settings**):
- **Internet: On** (needs a verified phone number).
- **Accelerator: GPU T4 x2** for a run of roughly 1 to 3 hours. With **None** it runs on the CPU and takes about 12 hours or more.
- Optional but recommended: **Add-ons → Secrets**, add `DRIVE_CHECKPOINT_URL` (your Google Drive saver's address, see `docs/benchmark/KAGGLE.md`) and tick it for this notebook. Progress and the log are then saved to your Google Drive every two minutes, and a new run carries on from the last save.

Then **Save Version → Save & Run All (Commit)**. It runs in the background, so you can close the tab. The run stops itself after 11 hours (Kaggle's limit is 12) so the results are always kept. The score is in the log's last line, in `humaneval-max.json` under **Output**, and in your Drive folder **Stick AI benchmark**.

**Carrying on after a stop:** with the Drive saver it's automatic. Without it, click **Add Input → Your Work**, pick this notebook's last version, then **Save & Run All** again.


In [ ]:
# Reads the optional Google Drive saver address from Kaggle's secrets (Add-ons → Secrets).
try:
    from kaggle_secrets import UserSecretsClient
    url = UserSecretsClient().get_secret("DRIVE_CHECKPOINT_URL") or ""
except Exception:
    url = ""
open("/tmp/drive-url", "w").write(url.strip())
print("Google Drive checkpoints:", "on" if url else "off (add the DRIVE_CHECKPOINT_URL secret to turn them on)")


In [ ]:
%%bash
set -euo pipefail
# 1 = all 164 problems. 4 = every 4th problem (41).
SAMPLE=1
LLAMA_COMMIT=cd959bae99ff4bed22bef2604f45241c09bbb8c8
HUMANEVAL_COMMIT=b9453e8f1865df0a0057a1383767caa400acfbcf
HUMANEVAL_SHA256=b796127e635a67f93fb35c04f4cb03cf06f38c8072ee7cee8833d7bee06979ef
OUT=/kaggle/working/humaneval-max.json
LOG=/kaggle/working/run.log
DRIVE=$(cat /tmp/drive-url 2>/dev/null || true)
WORK=/tmp/stick-bench; mkdir -p "$WORK"; cd "$WORK"
GPU=0; command -v nvidia-smi >/dev/null && nvidia-smi -L >/dev/null 2>&1 && GPU=1
echo "== Machine: $(nproc) cores, $(free -g | awk '/Mem/{print $2}') GB memory, GPU: $([ $GPU = 1 ] && nvidia-smi -L | tr '\n' ' ' || echo none)"

# Saves the results and the end of the log to Google Drive (if the saver is set up). Never stops the run.
save_to_drive() {
  [ -n "$DRIVE" ] || return 0
  [ -f "$OUT" ] && curl -fsSL -m 60 -X POST -H "Content-Type: text/plain" --data-binary @"$OUT" "$DRIVE?name=humaneval-max.json" >/dev/null 2>&1 || true
  [ -f "$LOG" ] && tail -c 500000 "$LOG" | curl -fsSL -m 60 -X POST -H "Content-Type: text/plain" --data-binary @- "$DRIVE?name=run.log" >/dev/null 2>&1 || true
}

echo "== Node.js"
curl -fsSL https://nodejs.org/dist/v24.22.1/node-v24.22.1-linux-x64.tar.xz | tar -xJ
export PATH="$WORK/node-v24.22.1-linux-x64/bin:$PATH"; node --version

echo "== Stick AI's benchmark script"
rm -rf stick_ai && git clone -q --depth 1 https://github.com/alfredmurray-goat/stick_ai.git
(cd stick_ai && npm install --ignore-scripts --no-audit --no-fund --loglevel=error && npm run build --silent)

# Carry on from an earlier run: Google Drive first, then an earlier version's Output added as input.
if [ ! -f "$OUT" ] && [ -n "$DRIVE" ]; then
  curl -fsSL -m 60 "$DRIVE?name=humaneval-max.json" -o /tmp/previous.json 2>/dev/null || true
  if [ -s /tmp/previous.json ] && node -e 'const d=JSON.parse(require("fs").readFileSync(process.argv[1],"utf8")); if(!Array.isArray(d.results)) process.exit(1)' /tmp/previous.json 2>/dev/null; then cp /tmp/previous.json "$OUT"; fi
fi
PREV=$(find /kaggle/input -name humaneval-max.json 2>/dev/null | head -1 || true)
if [ ! -f "$OUT" ] && [ -n "$PREV" ]; then cp "$PREV" "$OUT"; fi
[ -f "$OUT" ] && echo "== Carrying on: $(node -e 'console.log(require(process.argv[1]).results.length)' "$OUT") problems already done"

echo "== llama.cpp at the app's pinned version ($([ $GPU = 1 ] && echo 'with CUDA, about 20 minutes' || echo 'about 10 minutes'))"
# GGML_CUDA_NO_VMM: Kaggle's image has the CUDA toolkit but not the driver library to link against (CUDA::cuda_driver).
CUDA_FLAGS=""; [ $GPU = 1 ] && CUDA_FLAGS="-DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=75 -DGGML_CUDA_NO_VMM=ON"
if [ ! -x llama-build/bin/llama-server ]; then
  rm -rf llama.cpp && git init -q llama.cpp && git -C llama.cpp remote add origin https://github.com/ggml-org/llama.cpp.git
  git -C llama.cpp fetch -q --depth 1 origin "$LLAMA_COMMIT" && git -C llama.cpp checkout -q --detach FETCH_HEAD
  cmake -S llama.cpp -B llama-build -DCMAKE_BUILD_TYPE=Release -DBUILD_SHARED_LIBS=OFF -DGGML_NATIVE=OFF -DGGML_OPENMP=OFF -DGGML_METAL=OFF -DGGML_BLAS=OFF -DLLAMA_OPENSSL=OFF -DLLAMA_BUILD_TESTS=OFF -DLLAMA_BUILD_EXAMPLES=OFF -DLLAMA_BUILD_SERVER=ON -DLLAMA_USE_PREBUILT_UI=OFF -DLLAMA_BUILD_UI=OFF $CUDA_FLAGS > cmake.log 2>&1 || { tail -30 cmake.log; exit 1; }
  cmake --build llama-build --config Release --target llama-server -j "$(nproc)" > build.log 2>&1 || { tail -30 build.log; exit 1; }
fi

echo "== Stellan-1 Max (14.3 GB, checked against its SHA-256)"
read -r url sha < <(cd stick_ai && node -e 'import("./dist/portable/max-release.js").then(m => console.log(m.maxRelease.url + " " + m.bundledMax.sha256))')
[ -f max.gguf ] || curl -fL --retry 5 -sS -o max.gguf "$url"
echo "$sha  max.gguf" | sha256sum -c -

echo "== HumanEval"
curl -fsSL -o HumanEval.jsonl.gz "https://raw.githubusercontent.com/openai/human-eval/$HUMANEVAL_COMMIT/data/HumanEval.jsonl.gz"
echo "$HUMANEVAL_SHA256  HumanEval.jsonl.gz" | sha256sum -c - && gunzip -f HumanEval.jsonl.gz

echo "== Running (each line is one problem; the running score is in brackets)"
( while sleep 120; do save_to_drive; done ) & SAVER=$!
cd stick_ai
# Stop at 11 hours so Kaggle (12) never cuts the run off and the results are kept.
timeout 11h node scripts/benchmark-humaneval.mjs --server ../llama-build/bin/llama-server --gguf ../max.gguf --model stellan-1-max --problems ../HumanEval.jsonl --out "$OUT" --sample "$SAMPLE" --gpu-layers "$([ $GPU = 1 ] && echo 999 || echo 0)" 2>&1 | tee -a "$LOG" || echo "== Stopped at the 11-hour limit; open this notebook again and Save & Run All to carry on." | tee -a "$LOG"
kill $SAVER 2>/dev/null || true
node -e 'const d=require(process.argv[1]);const p=d.results.filter(r=>r.passed).length,n=d.results.length;const t=d.results.map(r=>r.genTokPerSec).filter(Boolean);console.log(`== RESULT: ${p}/${n} passed (${(100*p/n).toFixed(1)}%), ${n<164?"partial: run again to carry on":"complete"}; median ${t.sort((a,b)=>a-b)[t.length>>1]?.toFixed?.(1)??"?"} tokens/s`)' "$OUT" | tee -a "$LOG"
save_to_drive
